# 02 — Momentum Strategy Analysis

This notebook analyses the Jegadeesh-Titman (1993) cross-sectional momentum strategy
implemented in `strategies/momentum.py`.

**Sections**
- Baseline run & performance metrics
- Equity curve & drawdown
- Signal visualisation & momentum distribution
- Lookback sensitivity analysis (3, 6, 9, 12, 18, 24 months)
- Long-portfolio size sensitivity (n_long = 1, 3, 5, 7, 10)
- Ranked vs continuous signal comparison
- Automated findings

**Reference**: Jegadeesh, N. & Titman, S. (1993). *Returns to Buying Winners and Selling
Losers: Implications for Stock Market Efficiency.* Journal of Finance, 48(1), 65–91.

In [ ]:
%matplotlib inline
import sys
import warnings
import pathlib

_here = pathlib.Path(".").resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from backtester import DataLoader, Backtester, compute_metrics
from strategies.momentum import MomentumStrategy
from config import DEFAULT_TICKERS, BENCHMARK_TICKER, START_DATE, END_DATE, INITIAL_CAPITAL

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 5)

print("Environment ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Load price data — `prices` is the canonical wide-format DataFrame used
# throughout this notebook (index=DatetimeIndex, columns=ticker symbols).
# ---------------------------------------------------------------------------
all_tickers = list(DEFAULT_TICKERS) + [BENCHMARK_TICKER]

loader  = DataLoader(tickers=all_tickers, start_date=START_DATE, end_date=END_DATE)
prices  = loader.load_wide()   # <-- canonical variable name for this notebook
returns = loader.get_returns()

n_tickers = len(prices.columns)
n_days    = len(prices)
print(f"prices: {n_tickers} tickers × {n_days} trading days")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail(3)

In [ ]:
# ---------------------------------------------------------------------------
# Instantiate the baseline Jegadeesh-Titman strategy (12-1 rule, 5 longs)
# and run a single backtest.  All metrics are computed automatically inside
# Backtester.run().
# ---------------------------------------------------------------------------
strategy = MomentumStrategy(
    lookback_months=12,
    skip_months=1,
    n_long=5,
    n_short=0,
    signal_type="ranked",
)
print("Strategy:", strategy.get_name())

engine = Backtester(
    prices,
    config={
        "initial_capital": INITIAL_CAPITAL,
        "commission_bps":  5,
        "slippage_bps":    2,
        "position_sizing": "equal_weight",
        "benchmark":       BENCHMARK_TICKER,
    },
)

result = engine.run(strategy)
m      = result.metrics
print(f"\nBacktest complete — {len(result.trades)} trades executed.")

In [ ]:
# ---------------------------------------------------------------------------
# Print a formatted metrics table.
# ---------------------------------------------------------------------------
def _fmt(v):
    if v is None:              return "N/A"
    if isinstance(v, int):     return f"{v:,}"
    if isinstance(v, float):   return f"{v:.4f}"
    return str(v)

SEP = "-" * 50
print(f"{'Metric':<40}  {'Value':>10}")
print(SEP)
groups = [
    ("Returns", ["total_return_pct", "annualized_return_pct",
                 "annualized_volatility_pct", "best_day_pct",
                 "worst_day_pct", "positive_days_pct"]),
    ("Risk-adjusted", ["sharpe_ratio", "sortino_ratio", "calmar_ratio"]),
    ("Drawdown", ["max_drawdown_pct", "max_drawdown_duration_days", "recovery_days"]),
    ("Activity", ["n_trades", "win_rate_pct", "avg_trade_duration_days",
                  "total_cost_pct", "turnover_annual_pct"]),
    ("vs Benchmark", ["alpha_pct", "beta", "information_ratio",
                      "correlation_with_benchmark",
                      "benchmark_annualized_return_pct",
                      "benchmark_sharpe_ratio"]),
]
for section, keys in groups:
    print(f"  {section}")
    for k in keys:
        if k in m:
            print(f"    {k:<38}  {_fmt(m[k]):>10}")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Equity curve vs. benchmark + drawdown chart.
# ---------------------------------------------------------------------------
from backtester.metrics import drawdown_series

bench_result  = engine.run_benchmark()
bench_equity  = bench_result.equity_curve

strat_norm = result.equity_curve / result.equity_curve.iloc[0] * 100
bench_norm = bench_equity        / bench_equity.iloc[0]        * 100
dd         = drawdown_series(result.equity_curve)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

# ── Top: normalised equity curves ─────────────────────────────────────────
axes[0].plot(strat_norm.index, strat_norm.values,
             color="steelblue", linewidth=1.8, label=strategy.get_name())
axes[0].plot(bench_norm.index, bench_norm.values,
             color="black",     linewidth=1.4, linestyle="--",
             label=f"{BENCHMARK_TICKER} (buy & hold)")
axes[0].set_ylabel("Portfolio value (base = 100)")
axes[0].set_title("Equity curve vs. benchmark", fontweight="bold")
axes[0].legend(fontsize=9)

# ── Bottom: drawdown with fill_between ────────────────────────────────────
axes[1].fill_between(dd.index, dd.values, 0,
                     color="firebrick", alpha=0.4, label="Strategy drawdown")
axes[1].plot(dd.index, dd.values, color="firebrick", linewidth=0.8)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown", fontweight="bold")
axes[1].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Signal heatmap — last 756 trading days (≈ 3 years).
# Rows = dates, columns = tickers; colour = signal value.
# ---------------------------------------------------------------------------
raw_signals = strategy.generate_signals(prices)
heatmap_data = raw_signals.iloc[-756:].T  # tickers as rows, dates as columns

# Subsample dates to ~40 columns for readability
step = max(1, len(heatmap_data.columns) // 40)
hm_sub = heatmap_data.iloc[:, ::step]

# Format x-tick labels as YYYY-MM
xlabels = [d.strftime("%Y-%m") for d in hm_sub.columns]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    hm_sub,
    ax=ax,
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.0,
    xticklabels=xlabels,
    yticklabels=hm_sub.index.tolist(),
    cbar_kws={"label": "Signal value"},
)
ax.set_title(
    f"Momentum signal heatmap — last {len(raw_signals.iloc[-756:])} trading days",
    fontweight="bold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Ticker")
ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Momentum return distribution — cross-sectional distribution of the raw
# 12-month momentum returns across all tickers and all post-warmup dates.
# ---------------------------------------------------------------------------
from scipy import stats

# Recompute raw momentum returns (not yet ranked)
monthly = prices.resample("ME").last()
mom_monthly = monthly.pct_change(12).shift(1)
mom_daily   = mom_monthly.reindex(prices.index, method="ffill")

# Drop the warmup rows and flatten
post_warmup_mom = mom_daily.iloc[300:]
flat_mom = post_warmup_mom.values.flatten()
flat_mom = flat_mom[~np.isnan(flat_mom)] * 100  # convert to percent

mu_m    = float(np.mean(flat_mom))
sigma_m = float(np.std(flat_mom))
skew_m  = float(stats.skew(flat_mom))
kurt_m  = float(stats.kurtosis(flat_mom))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: histogram of cross-sectional momentum returns
sns.histplot(flat_mom, bins=80, kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(mu_m, color="blue",   linestyle="--", linewidth=1.8,
                label=f"Mean = {mu_m:.1f}%")
axes[0].axvline(mu_m + sigma_m, color="red", linestyle=":", linewidth=1.2)
axes[0].axvline(mu_m - sigma_m, color="red", linestyle=":", linewidth=1.2,
                label="±1 sigma")
stats_txt = (
    f"mu    = {mu_m:.2f}%\n"
    f"sigma = {sigma_m:.2f}%\n"
    f"skew  = {skew_m:.3f}\n"
    f"kurt  = {kurt_m:.3f}"
)
axes[0].text(0.97, 0.95, stats_txt, transform=axes[0].transAxes,
             va="top", ha="right", fontsize=9,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
axes[0].set_title("Cross-sectional momentum return distribution", fontweight="bold")
axes[0].set_xlabel("12-month momentum return (%)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)

# Right: per-ticker boxplot of momentum returns (last 756 days)
box_data = {col: post_warmup_mom[col].dropna() * 100
            for col in post_warmup_mom.columns}
sorted_tickers = sorted(box_data, key=lambda t: float(box_data[t].median()))
axes[1].boxplot([box_data[t].values for t in sorted_tickers],
                labels=sorted_tickers, vert=True,
                patch_artist=True,
                boxprops=dict(facecolor="lightsteelblue", color="steelblue"),
                medianprops=dict(color="firebrick", linewidth=1.5),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_title("Per-ticker momentum return distribution", fontweight="bold")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("12-month momentum return (%)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Lookback sensitivity — run the strategy for each lookback window and
# collect key metrics.
# ---------------------------------------------------------------------------
lookback_grid = [3, 6, 9, 12, 18, 24]
lookback_rows = []

for lb in lookback_grid:
    s   = MomentumStrategy(lookback_months=lb, skip_months=1,
                           n_long=5, signal_type="ranked")
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(s)
    mtr = r.metrics
    lookback_rows.append({
        "lookback_months":         lb,
        "total_return_pct":        mtr["total_return_pct"],
        "annualized_return_pct":   mtr["annualized_return_pct"],
        "annualized_volatility_pct": mtr["annualized_volatility_pct"],
        "sharpe_ratio":            mtr["sharpe_ratio"],
        "max_drawdown_pct":        mtr["max_drawdown_pct"],
        "calmar_ratio":            mtr["calmar_ratio"],
        "alpha_pct":               mtr.get("alpha_pct"),
    })
    print(f"  lookback={lb:>2}m  Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%  "
          f"MaxDD={mtr['max_drawdown_pct']:.1f}%")

df_lb = pd.DataFrame(lookback_rows).set_index("lookback_months")
df_lb

In [ ]:
# ---------------------------------------------------------------------------
# Lookback sensitivity chart.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = df_lb.index.tolist()

# Left: Sharpe ratio
axes[0].bar(x, df_lb["sharpe_ratio"], color="steelblue", width=1.8)
axes[0].axhline(0, color="black", linewidth=0.8)
for xi, yi in zip(x, df_lb["sharpe_ratio"]):
    axes[0].text(xi, yi + 0.01, f"{yi:.2f}", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Sharpe ratio by lookback window", fontweight="bold")
axes[0].set_xlabel("Lookback (months)")
axes[0].set_ylabel("Sharpe ratio")
axes[0].set_xticks(x)

# Middle: Annualised return vs volatility
axes[1].scatter(df_lb["annualized_volatility_pct"],
                df_lb["annualized_return_pct"],
                s=80, color="steelblue", zorder=3)
for lb_val, row in df_lb.iterrows():
    axes[1].annotate(
        f"{lb_val}m",
        xy=(row["annualized_volatility_pct"], row["annualized_return_pct"]),
        xytext=(4, 4), textcoords="offset points", fontsize=8,
    )
axes[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_title("Risk-return by lookback window", fontweight="bold")
axes[1].set_xlabel("Annualized volatility (%)")
axes[1].set_ylabel("Annualized return (%)")

# Right: Max drawdown
axes[2].bar(x, df_lb["max_drawdown_pct"], color="firebrick", width=1.8)
for xi, yi in zip(x, df_lb["max_drawdown_pct"]):
    axes[2].text(xi, yi - 0.5, f"{yi:.1f}%", ha="center", va="top", fontsize=8)
axes[2].set_title("Max drawdown by lookback window", fontweight="bold")
axes[2].set_xlabel("Lookback (months)")
axes[2].set_ylabel("Max drawdown (%)")
axes[2].set_xticks(x)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# n_long sensitivity — vary the number of long positions held.
# ---------------------------------------------------------------------------
nlong_grid = [1, 3, 5, 7, 10]
nlong_rows = []

for nl in nlong_grid:
    s   = MomentumStrategy(lookback_months=12, skip_months=1,
                           n_long=nl, signal_type="ranked")
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(s)
    mtr = r.metrics
    nlong_rows.append({
        "n_long":                    nl,
        "total_return_pct":          mtr["total_return_pct"],
        "annualized_return_pct":     mtr["annualized_return_pct"],
        "annualized_volatility_pct": mtr["annualized_volatility_pct"],
        "sharpe_ratio":              mtr["sharpe_ratio"],
        "max_drawdown_pct":          mtr["max_drawdown_pct"],
        "turnover_annual_pct":       mtr["turnover_annual_pct"],
        "alpha_pct":                 mtr.get("alpha_pct"),
    })
    print(f"  n_long={nl:>2}  Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%  "
          f"Turnover={mtr['turnover_annual_pct']:.1f}%")

df_nl = pd.DataFrame(nlong_rows).set_index("n_long")
df_nl

In [ ]:
# ---------------------------------------------------------------------------
# n_long sensitivity chart.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

xn = df_nl.index.tolist()

# Left: Sharpe ratio
colors_nl = ["steelblue" if v >= 0 else "firebrick"
             for v in df_nl["sharpe_ratio"]]
axes[0].bar(xn, df_nl["sharpe_ratio"], color=colors_nl, width=0.7)
axes[0].axhline(0, color="black", linewidth=0.8)
for xi, yi in zip(xn, df_nl["sharpe_ratio"]):
    axes[0].text(xi, yi + 0.01, f"{yi:.2f}", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Sharpe ratio by n_long", fontweight="bold")
axes[0].set_xlabel("Number of long positions")
axes[0].set_ylabel("Sharpe ratio")
axes[0].set_xticks(xn)

# Middle: return vs turnover scatter
sc = axes[1].scatter(
    df_nl["turnover_annual_pct"],
    df_nl["annualized_return_pct"],
    c=df_nl["sharpe_ratio"],
    cmap="RdYlGn",
    s=120,
    zorder=3,
)
for nl_val, row in df_nl.iterrows():
    axes[1].annotate(
        f"n={nl_val}",
        xy=(row["turnover_annual_pct"], row["annualized_return_pct"]),
        xytext=(4, 4), textcoords="offset points", fontsize=8,
    )
plt.colorbar(sc, ax=axes[1], label="Sharpe ratio")
axes[1].set_title("Return vs turnover by n_long", fontweight="bold")
axes[1].set_xlabel("Annual turnover (%)")
axes[1].set_ylabel("Annualized return (%)")

# Right: max drawdown
axes[2].bar(xn, df_nl["max_drawdown_pct"], color="firebrick", width=0.7)
for xi, yi in zip(xn, df_nl["max_drawdown_pct"]):
    axes[2].text(xi, yi - 0.5, f"{yi:.1f}%", ha="center", va="top", fontsize=8)
axes[2].set_title("Max drawdown by n_long", fontweight="bold")
axes[2].set_xlabel("Number of long positions")
axes[2].set_ylabel("Max drawdown (%)")
axes[2].set_xticks(xn)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Ranked vs continuous signal comparison.
# Run both variants with identical parameters and compare equity curves
# and summary metrics side-by-side.
# ---------------------------------------------------------------------------
variants = {
    "ranked":     MomentumStrategy(lookback_months=12, skip_months=1,
                                    n_long=5, signal_type="ranked"),
    "continuous": MomentumStrategy(lookback_months=12, skip_months=1,
                                    n_long=5, signal_type="continuous"),
}

variant_results = {}
comparison_rows = []

for name, strat in variants.items():
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(strat)
    variant_results[name] = r
    mtr = r.metrics
    comparison_rows.append({
        "signal_type":               name,
        "total_return_pct":          mtr["total_return_pct"],
        "annualized_return_pct":     mtr["annualized_return_pct"],
        "annualized_volatility_pct": mtr["annualized_volatility_pct"],
        "sharpe_ratio":              mtr["sharpe_ratio"],
        "max_drawdown_pct":          mtr["max_drawdown_pct"],
        "turnover_annual_pct":       mtr["turnover_annual_pct"],
        "alpha_pct":                 mtr.get("alpha_pct"),
    })
    print(f"  [{name:10s}]  "
          f"Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%  "
          f"MaxDD={mtr['max_drawdown_pct']:.1f}%  "
          f"Turnover={mtr['turnover_annual_pct']:.1f}%")

df_cmp = pd.DataFrame(comparison_rows).set_index("signal_type")
df_cmp

In [ ]:
# ---------------------------------------------------------------------------
# Ranked vs continuous: equity curve overlay + metrics bar chart.
# ---------------------------------------------------------------------------
palette = {"ranked": "steelblue", "continuous": "darkorange"}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: normalised equity curves
for name, r in variant_results.items():
    norm_eq = r.equity_curve / r.equity_curve.iloc[0] * 100
    axes[0].plot(norm_eq.index, norm_eq.values,
                 color=palette[name], linewidth=1.8, label=name)

# Add benchmark
bench_norm2 = bench_equity / bench_equity.iloc[0] * 100
axes[0].plot(bench_norm2.index, bench_norm2.values,
             color="black", linewidth=1.2, linestyle="--",
             label=f"{BENCHMARK_TICKER} (B&H)")
axes[0].set_title("Equity curves: ranked vs continuous", fontweight="bold")
axes[0].set_ylabel("Portfolio value (base = 100)")
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].legend(fontsize=9)

# Right: grouped bar chart of key metrics
metric_names = ["annualized_return_pct", "sharpe_ratio",
                "max_drawdown_pct",      "turnover_annual_pct"]
metric_labels = ["Ann. Return (%)", "Sharpe", "Max DD (%)", "Turnover (%)"]
x_pos    = np.arange(len(metric_names))
bar_w    = 0.35
variants_list = list(df_cmp.index)

for i, vname in enumerate(variants_list):
    vals = [df_cmp.loc[vname, mn] for mn in metric_names]
    bars = axes[1].bar(x_pos + i * bar_w, vals, bar_w,
                       label=vname, color=palette[vname], alpha=0.85)

axes[1].set_title("Metric comparison: ranked vs continuous", fontweight="bold")
axes[1].set_xticks(x_pos + bar_w / 2)
axes[1].set_xticklabels(metric_labels, rotation=15, ha="right", fontsize=9)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Automated Findings

In [ ]:
# ---------------------------------------------------------------------------
# Automated findings — derive and print key results programmatically so
# conclusions are always in sync with the actual numbers.
# ---------------------------------------------------------------------------
bench_mtr = bench_result.metrics if bench_result.metrics else compute_metrics(bench_result)
base_mtr  = result.metrics

# Best lookback
best_lb_idx  = df_lb["sharpe_ratio"].idxmax()
best_lb_sh   = float(df_lb.loc[best_lb_idx, "sharpe_ratio"])

# Best n_long
best_nl_idx  = df_nl["sharpe_ratio"].idxmax()
best_nl_sh   = float(df_nl.loc[best_nl_idx, "sharpe_ratio"])

# Best signal type
best_sig_idx = df_cmp["sharpe_ratio"].idxmax()
best_sig_sh  = float(df_cmp.loc[best_sig_idx, "sharpe_ratio"])

# Turnover difference
turn_ranked = float(df_cmp.loc["ranked",     "turnover_annual_pct"])
turn_cont   = float(df_cmp.loc["continuous", "turnover_annual_pct"])

SEP = "=" * 60
print(SEP)
print("BASELINE STRATEGY (lookback=12m, skip=1m, n_long=5, ranked)")
print(SEP)
print(f"  Total return        : {base_mtr['total_return_pct']:.2f}%")
print(f"  Annualized return   : {base_mtr['annualized_return_pct']:.2f}%")
print(f"  Sharpe ratio        : {base_mtr['sharpe_ratio']:.4f}")
print(f"  Max drawdown        : {base_mtr['max_drawdown_pct']:.2f}%")
if "alpha_pct" in base_mtr:
    print(f"  Alpha vs {BENCHMARK_TICKER:3s}        : {base_mtr['alpha_pct']:.2f}%")

print()
print(SEP)
print("LOOKBACK SENSITIVITY")
print(SEP)
print(f"  Best lookback       : {best_lb_idx} months  (Sharpe = {best_lb_sh:.4f})")
lb_sharpes = ', '.join(f"{lb}m={sh:.2f}"
                       for lb, sh in df_lb["sharpe_ratio"].items())
print(f"  All Sharpes         : {lb_sharpes}")

print()
print(SEP)
print("N_LONG SENSITIVITY")
print(SEP)
print(f"  Best n_long         : {best_nl_idx}  (Sharpe = {best_nl_sh:.4f})")
nl_sharpes = ', '.join(f"n={nl}:{sh:.2f}"
                       for nl, sh in df_nl["sharpe_ratio"].items())
print(f"  All Sharpes         : {nl_sharpes}")

print()
print(SEP)
print("RANKED vs CONTINUOUS")
print(SEP)
print(f"  Best signal type    : {best_sig_idx}  (Sharpe = {best_sig_sh:.4f})")
print(f"  Turnover (ranked)   : {turn_ranked:.1f}%")
print(f"  Turnover (continuous): {turn_cont:.1f}%")
turn_diff = turn_cont - turn_ranked
direction = "higher" if turn_diff > 0 else "lower"
print(f"  Continuous is {abs(turn_diff):.1f}pp {direction} turnover than ranked.")

## Notes

_Fill in manual observations after running the notebook:_

- Best-performing lookback window:
- Concentration vs diversification (n_long) trade-off:
- Ranked vs continuous — which wins and why:
- Observed turnover regime and cost sensitivity:
- Any anomalies or unexpected results: